# Insurance Exposure Dashboard — FEMA Flood Zone Overlay

Overlay a sample book of insured property locations with FEMA flood zone
polygons along the Orange County / Long Beach coast, assign a **risk tier**
to each policy via a spatial join, and export two GeoJSON FeatureCollections
ready for a color-coded frontend map layer:

1. `insured_properties.geojson` — point features, one per policy, with
   `risk_tier`, `fema_zone`, `insured_value`, and `annual_premium`.
2. `fema_flood_zones.geojson` — dissolved flood zone polygons for the study
   area, one feature per zone, to render as a color-coded background layer.

## FEMA Risk Tier Mapping

| Tier | FEMA Zones | Meaning |
|---|---|---|
| **Extreme** | VE, V | Coastal high hazard (base flood + wave action) |
| **High** | A, AE, AO, AH, A99 | 1% annual chance flood (100-year) |
| **Moderate** | X subtype `0.2 PCT ANNUAL CHANCE` | 500-year floodplain |
| **Levee-Protected** | X subtype `LEVEE` | Reduced risk due to levee |
| **Minimal** | X (other) | Outside mapped flood hazard |
| **Undetermined** | D | Not studied |

## 1. Import Required Libraries

In [ ]:
from sedona.spark import *
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import json

## 2. Configure Wherobots Session

In [ ]:
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

## 3. Define the Insured Property Portfolio

A sample book of coastal Southern California policies spanning Huntington
Beach, Sunset Beach, Newport Beach, and Long Beach — chosen to exercise a
mix of flood zones (VE, AE, X-levee, X-500yr).

In [ ]:
portfolio = [
    ("POL-001", "Huntington Beach Pier Residence", -117.9988, 33.6553, 1850000.0,  6200.0),
    ("POL-002", "Sunset Beach Bungalow",           -118.0709, 33.7192, 1250000.0,  4100.0),
    ("POL-003", "Newport Balboa Peninsula",        -117.9022, 33.6019, 3400000.0, 11200.0),
    ("POL-004", "Long Beach Belmont Shore",        -118.1274, 33.7589,  980000.0,  3500.0),
    ("POL-005", "Newport Corona del Mar",          -117.8740, 33.5950, 4200000.0, 14800.0),
    ("POL-006", "Seal Beach Cottage",              -118.1058, 33.7410,  875000.0,  3100.0),
    ("POL-007", "Huntington Harbour Waterfront",   -118.0690, 33.7233, 2150000.0,  7400.0),
    ("POL-008", "Costa Mesa Inland",               -117.9186, 33.6411,  740000.0,  2200.0),
]

schema = StructType([
    StructField("policy_id",       StringType()),
    StructField("property_name",   StringType()),
    StructField("lon",             DoubleType()),
    StructField("lat",             DoubleType()),
    StructField("insured_value",   DoubleType()),
    StructField("annual_premium",  DoubleType()),
])

portfolio_df = sedona.createDataFrame(portfolio, schema) \
    .withColumn("geometry", f.expr("ST_Point(lon, lat)"))

portfolio_df.createOrReplaceTempView("insured_properties")
portfolio_df.show(truncate=False)

## 4. Assign Risk Tier via Spatial Join

Join each property point to the Regrid parcel polygon it falls within, read
the embedded `fema_flood_zone` designation, and derive a human-readable
`risk_tier` label for color coding.

In [ ]:
PARCEL_TABLE = "wherobots_open_data.partner_samples.regrid_parcels"

properties_with_risk = sedona.sql(f"""
    SELECT
        p.policy_id,
        p.property_name,
        p.lon,
        p.lat,
        p.insured_value,
        p.annual_premium,
        p.geometry,
        COALESCE(r.fema_flood_zone, 'UNKNOWN')         AS fema_zone,
        COALESCE(r.fema_flood_zone_subtype, '')         AS fema_subtype,
        CASE
            WHEN r.fema_flood_zone IN ('VE','V')                   THEN 'Extreme'
            WHEN r.fema_flood_zone IN ('A','AE','AO','AH','A99')   THEN 'High'
            WHEN r.fema_flood_zone = 'X'
                 AND r.fema_flood_zone_subtype = '0.2 PCT ANNUAL CHANCE FLOOD HAZARD'
                                                                   THEN 'Moderate'
            WHEN r.fema_flood_zone = 'X'
                 AND r.fema_flood_zone_subtype LIKE '%LEVEE%'     THEN 'Levee-Protected'
            WHEN r.fema_flood_zone = 'X'                           THEN 'Minimal'
            WHEN r.fema_flood_zone = 'D'                           THEN 'Undetermined'
            ELSE 'Unknown'
        END AS risk_tier
    FROM insured_properties p
    LEFT JOIN {PARCEL_TABLE} r
      ON ST_Intersects(p.geometry, r.geometry)
      AND r.state2 = 'CA'
""").cache()

properties_with_risk.select(
    "policy_id", "property_name", "fema_zone", "fema_subtype", "risk_tier",
    "insured_value", "annual_premium"
).show(truncate=False)

## 5. Dissolve FEMA Flood Zone Polygons for the Study Area

Build the color-coded background layer. A small bounding box around the
portfolio limits the parcel scan; `ST_Union_Aggr` dissolves parcels sharing
the same FEMA zone into a single multipolygon;
`ST_SimplifyPreserveTopology` keeps the GeoJSON payload small enough to
ship to the frontend.

In [ ]:
STUDY_BBOX_WKT = (
    "POLYGON (("
    "-118.15 33.59, -117.85 33.59, -117.85 33.78, -118.15 33.78, -118.15 33.59"
    "))"
)

flood_zones_df = sedona.sql(f"""
    SELECT
        fema_flood_zone AS fema_zone,
        CASE
            WHEN fema_flood_zone IN ('VE','V')                 THEN 'Extreme'
            WHEN fema_flood_zone IN ('A','AE','AO','AH','A99') THEN 'High'
        END AS risk_tier,
        COUNT(*) AS parcel_count,
        ST_SimplifyPreserveTopology(
            ST_Union_Aggr(geometry), 0.0005
        ) AS geometry
    FROM {PARCEL_TABLE}
    WHERE state2 = 'CA'
      AND ST_Intersects(
          geometry,
          ST_GeomFromText('{STUDY_BBOX_WKT}')
      )
      AND fema_flood_zone IN ('VE','V','A','AE','AO','AH','A99')
    GROUP BY fema_flood_zone
""").cache()

flood_zones_df.select("fema_zone", "risk_tier", "parcel_count").show(truncate=False)

## 6. Export Insured Properties as GeoJSON

One `Point` feature per policy, with the `risk_tier` color-coding field and
the underlying FEMA attributes preserved for tooltips.

In [ ]:
property_rows = properties_with_risk \
    .withColumn("geojson", f.expr("ST_AsGeoJSON(geometry)")) \
    .select(
        "policy_id", "property_name", "lon", "lat",
        "insured_value", "annual_premium",
        "fema_zone", "fema_subtype", "risk_tier", "geojson"
    ).collect()

property_features = [
    {
        "type": "Feature",
        "properties": {
            "policy_id":      row.policy_id,
            "property_name":  row.property_name,
            "insured_value":  row.insured_value,
            "annual_premium": row.annual_premium,
            "fema_zone":      row.fema_zone,
            "fema_subtype":   row.fema_subtype,
            "risk_tier":      row.risk_tier,
        },
        "geometry": json.loads(row.geojson),
    }
    for row in property_rows
]

property_fc = {"type": "FeatureCollection", "features": property_features}

properties_out = "/tmp/insured_properties.geojson"
with open(properties_out, "w") as fh:
    json.dump(property_fc, fh)

print(f"Wrote {len(property_features)} policy points to {properties_out}")
print(json.dumps(property_fc, indent=2)[:1200], "...")

## 7. Export FEMA Flood Zone Polygons as GeoJSON

One `MultiPolygon` feature per FEMA zone, each tagged with its `risk_tier`
so the frontend can style them consistently with the property points (e.g.
red fill for Extreme/VE, orange for High/AE).

In [ ]:
zone_rows = flood_zones_df \
    .withColumn("geojson", f.expr("ST_AsGeoJSON(geometry)")) \
    .select("fema_zone", "risk_tier", "parcel_count", "geojson") \
    .collect()

zone_features = [
    {
        "type": "Feature",
        "properties": {
            "fema_zone":    row.fema_zone,
            "risk_tier":    row.risk_tier,
            "parcel_count": row.parcel_count,
        },
        "geometry": json.loads(row.geojson),
    }
    for row in zone_rows
]

zone_fc = {"type": "FeatureCollection", "features": zone_features}

zones_out = "/tmp/fema_flood_zones.geojson"
with open(zones_out, "w") as fh:
    json.dump(zone_fc, fh)

print(f"Wrote {len(zone_features)} dissolved flood zones to {zones_out}")
for feat in zone_features:
    props = feat["properties"]
    print(f"  {props['fema_zone']:<4} {props['risk_tier']:<8} {props['parcel_count']:>6} parcels")

## 8. Portfolio Exposure Summary

Aggregate insured value and premium by `risk_tier` — the rollup the
dashboard header can display alongside the map.

In [ ]:
properties_with_risk.createOrReplaceTempView("scored_properties")

sedona.sql("""
    SELECT
        risk_tier,
        COUNT(*)                         AS policy_count,
        ROUND(SUM(insured_value))        AS total_insured_value,
        ROUND(SUM(annual_premium))       AS total_annual_premium,
        ROUND(AVG(insured_value))        AS avg_insured_value
    FROM scored_properties
    GROUP BY risk_tier
    ORDER BY
        CASE risk_tier
            WHEN 'Extreme'         THEN 1
            WHEN 'High'            THEN 2
            WHEN 'Moderate'        THEN 3
            WHEN 'Levee-Protected' THEN 4
            WHEN 'Minimal'         THEN 5
            ELSE 6
        END
""").show(truncate=False)

## 9. Preview on a Map

Render both layers with `SedonaKepler` for a quick visual check before
plugging the GeoJSON files into the insurance exposure dashboard.

In [ ]:
map_view = SedonaKepler.create_map(
    flood_zones_df.select("fema_zone", "risk_tier", "geometry"),
    name="FEMA Flood Zones"
)
SedonaKepler.add_df(
    map_view,
    properties_with_risk.select(
        "policy_id", "property_name", "risk_tier",
        "insured_value", "geometry"
    ),
    name="Insured Properties"
)
map_view